In [20]:
import pickle
import pandas as pd
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [21]:
with open("pakiet/train.pkl", "rb") as f:
    train_data = pickle.load(f)

with open("pakiet/test_no_target.pkl", "rb") as f:
    test_data = pickle.load(f)


In [22]:
all_chords = []

for seq, _ in train_data:
    all_chords.extend(seq)

counter = Counter(all_chords)

vocab = {
    chord: idx + 1
    for idx, (chord, _) in enumerate(counter.items())
}

PAD_IDX = 0


In [23]:

class MusicDataset(Dataset):
    def __init__(self, data, vocab, test=False):
        self.data = data
        self.vocab = vocab
        self.test = test

    def __len__(self):
        return len(self.data)

    def encode(self, seq):
        return torch.tensor(
            [self.vocab.get(ch, 0) for ch in seq],
            dtype=torch.long
        )

    def __getitem__(self, idx):

        if self.test:
            seq = self.data[idx]
            return self.encode(seq)

        seq, target = self.data[idx]

        return self.encode(seq), target


train_dataset = MusicDataset(train_data, vocab)
test_dataset = MusicDataset(test_data, vocab, test=True)


In [24]:

def collate_fn(batch):

    if isinstance(batch[0], tuple):

        sequences = [x[0] for x in batch]
        targets = [x[1] for x in batch]

        lengths = torch.tensor([len(seq) for seq in sequences])

        padded = pad_sequence(
            sequences,
            batch_first=True,
            padding_value=PAD_IDX
        )

        targets = torch.tensor(targets)

        return padded, lengths, targets

    else:

        lengths = torch.tensor([len(seq) for seq in batch])

        padded = pad_sequence(
            batch,
            batch_first=True,
            padding_value=PAD_IDX
        )

        return padded, lengths


In [25]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

In [26]:

class ComposerClassifier(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim=128,
        hidden_dim=128,
        num_layers=2,
        num_classes=5
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=PAD_IDX
        )

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, lengths):

        embedded = self.embedding(x)

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, (hidden, _) = self.lstm(packed)

        out = hidden[-1]

        out = self.fc(out)

        return out


model = ComposerClassifier(
    vocab_size=len(vocab) + 1
).to(DEVICE)

In [27]:

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
EPOCHS = 1

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for x, lengths, y in train_loader:

        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(x, lengths)

        loss = criterion(outputs, y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: {total_loss:.4f}")


model.eval()

predictions = []

with torch.no_grad():

    for x, lengths in test_loader:

        x = x.to(DEVICE)

        outputs = model(x, lengths)

        preds = torch.argmax(outputs, dim=1)

        predictions.extend(preds.cpu().numpy())


Epoch 1: 96.0940


In [29]:
pd.DataFrame(predictions).to_csv(
    "pred.csv",
    index=False,
    header=False
)

print("Zapisano pred.csv")

Zapisano pred.csv
